## Data Preparation

You should prepare the following things before running this step. 

1. **trained model** from ```step3.ipynb``` 

2. **A patient list** that emunarates the dataset 
   - check ```step2.ipynb```
   - for example data: as we said in previous steps, we are going to validate our models on type 2 noise. so please use  ```example_data/Patient_lists/patient_list_unsupervised_gaussian.xlsx``` for unsupervised learning.

3. bins for **histogram equalization**
    - provided in ```/help_data```

---

## Task: Inference

- first: we do inference for multiple times (section "step 5.prediction" in the script) 
- second: we take the average (section "step6.average" in the sript) to be the final product.

---

### Docker environment
Please use `docker/docker_pytorch`, it will build a pytorch docker


In [22]:
import sys
sys.path.append('/host/c/Users/ROG/Documents/Github')
import os
import torch
import numpy as np 
import nibabel as nb
import CTDenoising_Diffusion_N2N.denoising_diffusion_pytorch.denoising_diffusion_pytorch.conditional_diffusion as ddpm
import CTDenoising_Diffusion_N2N.functions_collection as ff
import CTDenoising_Diffusion_N2N.Build_lists.Build_list as Build_list
import CTDenoising_Diffusion_N2N.Generator as Generator

main_path = '/host/c/Users/ROG/Documents/Github/CTDenoising_Diffusion_N2N/' # replace with your own path

### step 1: define trial name and trained model file

In [23]:
supervision = 'supervised' # 'unsupervised' or 'supervised'
noise_type = 'possion' if supervision == 'supervised' else 'gaussian'
beta = 0 # by default

trial_name = 'model_'+supervision + '_' + noise_type + '_beta' + str(beta)
print(trial_name)

model_supervised_possion_beta0


In [24]:
model_root = '/host/d/file/denoising/models'  # 对应 D:\file\denoising\models
epoch = 1500
trial_name = 'model_supervised_possion_beta0'

trained_model_filename = os.path.join(
    model_root,
    trial_name,
    'models',
    f'model-{epoch}.pt'
)

save_folder = os.path.join(main_path,'models', trial_name,'pred_images')
os.makedirs(save_folder, exist_ok=True)

### step 2: set default parameters
usually you don't need to change

In [25]:
problem_dimension = '2D'
condition_channel = 0
image_size = [512, 512]

objective = 'pred_x0'

histogram_equalization = True
background_cutoff = -1000
maximum_cutoff = 2000
normalize_factor = 'equation'


### step 3: define patient list
test on type 2 noise  (Gaussian noise)

In [26]:
build_sheet =  Build_list.Build(os.path.join(main_path, 'example_data/Patient_lists','/host/d/file/xingyi_datasets.xlsx'))
_,patient_id_list,patient_subid_list,random_num_list, condition_list, x0_list = build_sheet.__build__(batch_list = [0])  # for the purpose of example, we test on the same training case
# n = ff.get_X_numbers_in_interval(total_number = patient_id_list.shape[0],start_number = 0,end_number = 1, interval = 2) # each case has two simulations, we do on the first one as example
print('total number:', patient_id_list.shape[0])

total number: 1


### step 4: define model

In [27]:
model = ddpm.Unet(
    problem_dimension=problem_dimension,
    init_dim=64,
    out_dim=1,
    channels=1, 
    conditional_diffusion=False,
    condition_channels=condition_channel,
    downsample_list=(True, True, True, False),
    upsample_list=(True, True, True, False),
    full_attn=(None, None, False, True),
)

diffusion_model = ddpm.GaussianDiffusion(
    model,
    image_size=image_size,
    timesteps=2000,
    sampling_timesteps=250,
    ddim_sampling_eta=1.,
    force_ddim=False,
    auto_normalize=False,
    objective=objective,
    clip_or_not=True, 
    clip_range=[-1, 1], 
    beta_schedule='reverse_warmup',
)

is ddim sampling True


### step 5: Prediction (doing inference for multiple times)
to minimize data storage, we only evaluated on the middle 50 slices (slice 30 - 80)

In [ ]:
slice_range = [0, 50]
inference_times = 20

for i in range(0, patient_id_list.shape[0]):
    patient_id = patient_id_list[i]
    patient_subid = patient_subid_list[i]
    random_num = random_num_list[i]
    x0_file = x0_list[i]
    condition_file = condition_list[i]

    print(i, patient_id, patient_subid, random_num)

    # get the condition image (original noisy image)
    print('condition_file:', condition_file, 'shape:', nb.load(condition_file).get_fdata().shape)
    condition_img = nb.load(condition_file).get_fdata()[:, :, slice_range[0]:slice_range[1]]
    affine = nb.load(condition_file).affine
    shape = condition_img.shape

    # get the ground truth image
    gt_img = nb.load(x0_file)
    print('x0_file:', x0_file, 'shape:', gt_img.get_fdata().shape)
    gt_img = gt_img.get_fdata()[:, :, slice_range[0]:slice_range[1]]

    for iteration in range(1, 1 + inference_times):
        print('iteration:', iteration)

        # make folders
        ff.make_folder([
            os.path.join(save_folder, patient_id), 
            os.path.join(save_folder, patient_id, patient_subid), 
            os.path.join(save_folder, patient_id, patient_subid, 'random_' + str(random_num))
        ])
        save_folder_case = os.path.join(save_folder, patient_id, patient_subid, 'random_' + str(random_num), 'epoch' + str(epoch) + '_' + str(iteration))
        os.makedirs(save_folder_case, exist_ok=True)

        if os.path.isfile(os.path.join(save_folder_case, 'pred_img.nii.gz')):
            print('already done')
            continue

        # generator
        generator = Generator.Dataset_2D(
            supervision=supervision,
            y_bar_list=np.array([x0_file]),
            original_x_list=np.array([condition_file]),
            image_size=image_size,
            num_slices_per_image=slice_range[1] - slice_range[0],
            random_pick_slice=False,
            slice_range=[slice_range[0], slice_range[1]],
            histogram_equalization=histogram_equalization,
            bins=np.load('/host/d/file/histogram_equalization/bins.npy'),
            bins_mapped=np.load('/host/d/file/histogram_equalization/bins_mapped.npy'),
            background_cutoff=background_cutoff,
            maximum_cutoff=maximum_cutoff,
            normalize_factor=normalize_factor,
        )

        # sample
        sampler = ddpm.Sampler(diffusion_model, generator, batch_size=1)

        # ===== 【新增】测试：模型输出到底像谁 =====
        if iteration == 1:  # 只测试一次
            import torch.nn.functional as F
            
            # 加载模型
            sampler.load_model(trained_model_filename)
            device = sampler.device
            model = sampler.ema.ema_model
            model.eval()
            
            # 获取一个样本
            y_bar_sample = generator[0][0].unsqueeze(0).to(device)  # N2N 输出
            x_orig_sample = generator[0][1].unsqueeze(0).to(device)  # 原始噪声图像
            
            # 转换到 [-1, 1]
            if x_orig_sample.min() >= 0:  # 只有 [0, 1] 范围才需要转换
                x_orig_sample_norm = x_orig_sample * 2.0 - 1.0
                y_bar_sample_norm = y_bar_sample * 2.0 - 1.0
            else:  # 已经是 [-1, 1] 范围
                x_orig_sample_norm = x_orig_sample
                y_bar_sample_norm = y_bar_sample
            
            # 在测试代码里加这个
            print(f"x_orig_sample (原始) range: [{x_orig_sample.min():.4f}, {x_orig_sample.max():.4f}]")
            print(f"转换后 range: [{x_orig_sample_norm.min():.4f}, {x_orig_sample_norm.max():.4f}]")

            print(f"\n===== 模型输出测试 =====")
            print(f"x_orig range: [{x_orig_sample_norm.min():.4f}, {x_orig_sample_norm.max():.4f}]")
            print(f"y_bar range: [{y_bar_sample_norm.min():.4f}, {y_bar_sample_norm.max():.4f}]")
            print(f"x_orig vs y_bar MSE: {F.mse_loss(x_orig_sample_norm, y_bar_sample_norm).item():.6f}")
            
            # 测试不同的 t 值
            with torch.no_grad():
                for t_val in [1, 10, 33, 100]:
                    t = torch.tensor([t_val], device=device, dtype=torch.long)
                    
                    # 模型直接输出
                    output = model.model(x_orig_sample_norm, t)
                    
                    # 计算相似度
                    mse_to_original = F.mse_loss(output, x_orig_sample_norm).item()
                    mse_to_ybar = F.mse_loss(output, y_bar_sample_norm).item()
                    
                    closer_to = 'y_bar ✅' if mse_to_ybar < mse_to_original else 'original_X ⚠️'
                    
                    print(f"\nt = {t_val}:")
                    print(f"  输出 vs original_X: {mse_to_original:.6f}")
                    print(f"  输出 vs y_bar:      {mse_to_ybar:.6f}")
                    print(f"  更接近: {closer_to}")
            
            print(f"\n===== 测试结束 =====\n")
        # ===== 测试代码结束 =====

        # ===== 使用 DDM² 去噪方法 =====
        # 不需要传 num_steps，只需要 start_t（或让它自动计算）
        pred_img = sampler.sample_2D(
            trained_model_filename, 
            condition_img,
            start_t=1, # 自动 State Matching
            batch_size= 2
        )
        print(pred_img.shape)
    
        # save
        nb.save(nb.Nifti1Image(pred_img, affine), os.path.join(save_folder_case, 'pred_img.nii.gz'))
        if iteration == 1:
            nb.save(nb.Nifti1Image(condition_img, affine), os.path.join(save_folder_case, 'condition_img.nii.gz'))
            nb.save(nb.Nifti1Image(gt_img, affine), os.path.join(save_folder_case, 'gt_img.nii.gz'))

0 00214841 0000455418 0
condition_file: /host/d/file/pre/noise2noise/pred_images/00214841/0000455418/random_0/epoch78/condition_img.nii.gz shape: (512, 512, 50)
x0_file: /host/d/file/pre/noise2noise/pred_images/00214841/0000455418/random_0/epoch78/pred_img.nii.gz shape: (512, 512, 50)
iteration: 1
histogram equalization:  True
x_orig_sample (原始) range: [-1.0000, 0.8502]
转换后 range: [-3.0000, 0.7004]

===== 模型输出测试 =====
x_orig range: [-3.0000, 0.7004]
y_bar range: [-3.0000, 0.6812]
x_orig vs y_bar MSE: 0.006865

t = 1:
  输出 vs original_X: 0.008480
  输出 vs y_bar:      0.011928
  更接近: original_X ⚠️

t = 10:
  输出 vs original_X: 0.008510
  输出 vs y_bar:      0.011906
  更接近: original_X ⚠️

t = 33:
  输出 vs original_X: 0.008617
  输出 vs y_bar:      0.011838
  更接近: original_X ⚠️

t = 100:
  输出 vs original_X: 0.008564
  输出 vs y_bar:      0.011379
  更接近: original_X ⚠️

===== 测试结束 =====



DDM² Sampling:   0%|          | 0/25 [00:00<?, ?it/s]

DEBUG - x_orig range: [-1.0000, 0.8508]
DEBUG - y_bar range: [-1.0000, 0.8406]
DDM² denoising from t=1


DDM² Sampling:   4%|▍         | 1/25 [00:01<00:38,  1.61s/it]

DEBUG - output range: [0.0000, 0.9169]
DDM² denoising from t=1


...

DDM² denoising from t=1


DDM² Sampling:  12%|█▏        | 3/25 [00:03<00:26,  1.21s/it]

DDM² denoising from t=1


DDM² Sampling:  12%|█▏        | 3/25 [00:04<00:35,  1.63s/it]


KeyboardInterrupt: 

### step 6: average the results of multiple inferences

In [ ]:
slice_range = [0,50] # the range of slices to be used
inference_avg_scans = [10,20] # avg 10 or 20 inference results

for i in range(0,patient_id_list.shape[0]):
    patient_id = patient_id_list[i]
    patient_subid = patient_subid_list[i]
    random_num = random_num_list[i]
    x0_file = x0_list[i]
    condition_file = condition_list[i]

    print(i,patient_id, patient_subid, random_num)

    save_folder_avg = os.path.join(save_folder, patient_id, patient_subid, 'random_' + str(random_num), 'epoch' + str(epoch)+'avg'); os.makedirs(save_folder_avg, exist_ok=True)

    # get the condition image (original noisy image)
    print('condition_file:', condition_file, 'shape: ', nb.load(condition_file).get_fdata().shape)
    condition_img = nb.load(condition_file).get_fdata()[:,:,slice_range[0]:slice_range[1]]
    affine = nb.load(condition_file).affine
    shape = condition_img.shape
        
    made_predicts = ff.sort_timeframe(ff.find_all_target_files(['epoch' + str(epoch)+'_*'], os.path.join(save_folder, patient_id, patient_subid, 'random_' + str(random_num))),0,'_','/')
    print(made_predicts)
    total_predicts = len(made_predicts)

    loaded_data = np.zeros((shape[0], shape[1], shape[2], total_predicts))
    for j in range(total_predicts):
        loaded_data[:,:,:,j] = nb.load(os.path.join(made_predicts[j],'pred_img.nii.gz')).get_fdata()

    for avg_num in inference_avg_scans:
        print('avg_num:', avg_num)
        predicts_avg = np.zeros((shape[0], shape[1], shape[2], avg_num))
        print('predict_num:', avg_num)
        for j in range(avg_num):
            print('file:', made_predicts[j])
            predicts_avg[:,:,:,j] = loaded_data[:,:,:,j]
        # average across last axis
        predicts_avg = np.mean(predicts_avg, axis = -1)
        nb.save(nb.Nifti1Image(predicts_avg, affine), os.path.join(save_folder_avg, 'pred_img_scans' + str(avg_num) + '.nii.gz'))

0 00214841 0000455418 0
condition_file: /host/d/file/pre/noise2noise/pred_images/00214841/0000455418/random_0/epoch78/condition_img.nii.gz shape:  (512, 512, 50)
['/host/c/Users/ROG/Documents/Github/CTDenoising_Diffusion_N2N/models/model_supervised_possion_beta0/pred_images/00214841/0000455418/random_0/epoch2000_1'
 '/host/c/Users/ROG/Documents/Github/CTDenoising_Diffusion_N2N/models/model_supervised_possion_beta0/pred_images/00214841/0000455418/random_0/epoch2000_2'
 '/host/c/Users/ROG/Documents/Github/CTDenoising_Diffusion_N2N/models/model_supervised_possion_beta0/pred_images/00214841/0000455418/random_0/epoch2000_3'
 '/host/c/Users/ROG/Documents/Github/CTDenoising_Diffusion_N2N/models/model_supervised_possion_beta0/pred_images/00214841/0000455418/random_0/epoch2000_4'
 '/host/c/Users/ROG/Documents/Github/CTDenoising_Diffusion_N2N/models/model_supervised_possion_beta0/pred_images/00214841/0000455418/random_0/epoch2000_5'
 '/host/c/Users/ROG/Documents/Github/CTDenoising_Diffusion_N2N/